DISCLAIMER: Code template is from the Lab 2 file. I adjusted the template as needed to match the questions and requirements of the homework. The original model was not working and I reached out to a classmate for assistance and they helped me change the model to what is currently in the code ("openai/gpt-oss-20b"). I adjusted the token counts as needed for my outputs. I used Groq to create a new API key as instructed in class.

AI, specifically CoPilot, was used for Question 1 to assist with the coding script to help get my prompt to loop 5 times for teach temperature. The change was completed in the second section of the code, where the function is called. I used range() to repeatedly call the function five times and print each response. The 'i+1' was used because Python coding count starts at 0 instead of 1.

# 🧪 BIA 662: Lab 2 - Controlling the Ghost in the Machine
**Topic:** LLM Inference Parameters & Prompt Engineering

In this lab, we will look "under the hood" of Large Language Models. We aren't just chatting; we are controlling the **inference engine**. We will adjust parameters that control how the model "thinks" and selects the next word.

**Learning Objectives:**
1.  **Temperature:** Understand the trade-off between "Fact/Logic" (Low Temp) and "Creativity/Chaos" (High Temp).
2.  **System Prompts:** Learn how to "condition" the model to adopt a specific persona or output format (JSON).
3.  **Hallucinations:** Intentionally break the model to see how it fabricates information.
4.  **Determinism:** Use `seed` values to make AI outputs reproducible.

**Prerequisite:** You need a [Groq API Key](https://console.groq.com/keys).

In [ ]:
!pip install groq

In [ ]:
from groq import Groq

# API Key created for Lab 2 homework with Groq

client = Groq(api_key="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")

## 🌡️ Experiment 1: Temperature & The Logic Trap
**Concept:** `Temperature` controls the randomness of the model's output.
* **Low Temperature (0.0 - 0.3):** The model becomes deterministic. It picks the most likely next token. Good for math, code, and facts.
* **High Temperature (0.7 - 1.5):** The model takes risks. It picks less likely tokens. Good for poetry and brainstorming, but bad for logic.

**Lab 2 Question 1: The Temperature Experiment (Chaos vs. Order):**.
* Write a script to loop the same prompt (e.g., "Write a 1-sentence slogan for a coffee company") **5 times**.
* Run the loop first with temperature = 0.0 and then with temperature = 1.0.
* **Output:** Observe the variation. Explain in one sentence which setting is safer for a financial report and why.


In [ ]:
def run_chat_experiment(temp_setting):
    try:
        completion = client.chat.completions.create(
            # Using the fast, cheap model from your docs
            model="openai/gpt-oss-20b",
            messages=[
                {"role": "system", "content": "You are a helpful teaching assistant."},
                {"role": "user", "content": "Write a 1-sentence slogan for a coffee company."}
            ],
            temperature=temp_setting,
            max_tokens=500,
            reasoning_effort="low"
            #use more tokens but less effort so it isnt slow
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

In [ ]:
print("--- Temperature 0 (Logic) ---")
for i in range(5):
  print(f"{i+1}. {run_chat_experiment(0.0)}")

print("\n--- Temperature 1 (Creative) ---")
for i in range(5):
  print(f"{i+1}. {run_chat_experiment(1.0)}")

--- Temperature 0 (Logic) ---
1. "Awaken your day, one bold sip at a time."
2. "Awaken your day, one bold sip at a time."
3. "Awaken your day, one bold sip at a time."
4. "Awaken your day, one bold sip at a time."
5. "Awaken your day, one bold sip at a time."

--- Temperature 1 (Creative) ---
1. "Awaken your world, one bold sip at a time."
2. Brew the moments that keep you buzzing.
3. Brew the moment, one cup at a time.
4. Brew the day bright—one cup at a time.
5. "Fuel your day the aromatic way—one sip, endless possibilities."


Temperature 0.0 would be the safer option for a financial report. It is more consistent in it's responses, and produced similar if not exact outputs for each response. This would lessen the chances of inaccurate responses in a financial report. Temperature 1.0 produced more randomness in it's responses. This creates a risk of innacurate results in a financial report.

## 🤖 Experiment 3: The System Prompt (Conditioning)
**Concept:** The **System Prompt** is the "God Mode" instruction. It sets the behavior, rules, and boundaries *before* the user even speaks.

We use System Prompts for three main goals:
1.  **Safety:** preventing the model from doing illegal things.
2.  **Persona:** giving the model a personality (e.g., "You are a Caveman").
3.  **Formatting:** forcing the model to output machine-readable code (e.g., "Speak only JSON").

**Lab 2 Question 2: The System Prompt (The Persona Constraint):**
* **Task:** Ask the model: "How do I make a peanut butter sandwich?"
* **Constraint:** Use a **System prompt** to force the model to answer **ONLY** on valid JSON format with keys for "ingredients" and "step_count". It must not output any conversational text (no "Here is your JSON").
* **Output:** Print the raw JSON response.

In [ ]:
def experiment_system_prompt(system_role, user_query):
    model = "openai/gpt-oss-20b"
    try:
        completion = client.chat.completions.create(
            model= model,
            messages=[
                {"role": "system", "content": system_role},
                {"role": "user", "content": user_query}
            ],
            temperature=0.7,
            max_tokens= 500
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

# The Safe Query
query = "How do I make a peanut butter sandwich?"


# --- PERSONA 3: THE JSON MACHINE (Format Constraint) --- for Lab 2 Question 2
# This proves we can force the model to be a "Database"
print(f"\n--- 3. The JSON Parser ---")
print(experiment_system_prompt("You are a data extraction bot. Output ONLY valid JSON with keys: 'ingredients', 'steps_count'. ", query))


--- 3. The JSON Parser ---
{"ingredients":["bread","peanut butter"],"steps_count":3}
